# 🎬 Video Upscaler Pro no Google Colab (Tesla T4)
### Super-Resolução Neural (1080p / 4K) com Áudio Sincronizado para Clipes de 10s a 30s+

Este notebook é uma ferramenta autônoma dedicada ao **upscaling de alta fidelidade de vídeos**:
* 🚀 **Projetado para clipes de 10s a 30s+:** Processamento ultrarrápido na GPU Tesla T4 (~1 a 3 minutos por vídeo).
* 📁 **Modo Arquivo Único ou Modo Pasta (Lote):** Processe um único clipe ou aponte para uma pasta inteira no Google Drive e deixe ele processar toda a sua fila automaticamente.
* 🧠 **Modelos Top de Linha:** `4x-UltraSharp` (máxima nitidez fotorrealista) e `RealESRGAN_x4plus`.
* 🎵 **Áudio Estéreo 100% Sincronizado:** Isola e remonta as trilhas de som sem perda de sincronia.
* 💾 **Integração Total com o Google Drive:** Os modelos ficam salvos permanentemente e os vídeos finais são gravados direto no seu Drive.


In [ ]:
# =============================================================================
# 1. Verificação de GPU e Montagem do Google Drive
# =============================================================================
import os
import sys
import subprocess
import shutil
from pathlib import Path
import torch

print("🔍 1. Verificando Hardware...")
if not torch.cuda.is_available():
    raise RuntimeError("❌ Nenhuma GPU detectada! Vá em Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU T4.")
gpu_name = torch.cuda.get_device_name(0)
print(f"   GPU Ativa: {gpu_name} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)")

print("\n📁 2. Conectando ao Google Drive...")
DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

MY_DRIVE = DRIVE_MOUNT / "MyDrive"
print("   Google Drive montado com sucesso!")

print("\n📦 3. Instalando dependências necessárias (spandrel, opencv, tqdm)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spandrel", "opencv-python", "tqdm", "huggingface_hub"], check=True)
print("✅ Ambiente pronto para Upscaling!")


In [ ]:
# =============================================================================
# 2. Gerenciador de Modelos de Super-Resolução (Google Drive)
# =============================================================================
# Detecta a pasta ComfyUI ou cria a biblioteca permanente de modelos no Drive
drive_candidates = [
    MY_DRIVE / "ComfyUI" / "models" / "upscale_models",
    MY_DRIVE / "ComfyUi" / "models" / "upscale_models",
    MY_DRIVE / "comfyui" / "models" / "upscale_models",
    MY_DRIVE / "models" / "upscale_models",
]
UPSCALE_DIR = next((p for p in drive_candidates if p.is_dir()), MY_DRIVE / "ComfyUI" / "models" / "upscale_models")
UPSCALE_DIR.mkdir(parents=True, exist_ok=True)

UPSCALE_MODELS = {
    "4x-UltraSharp": {
        "file": "4x-UltraSharp.pth",
        "repo": "lokcx/4x-Ultrasharp",
        "bytes": 66986421,
        "desc": "UltraSharp (Recomendado - Nitidez cinematográfica e texturas realistas)"
    },
    "RealESRGAN_x4plus": {
        "file": "RealESRGAN_x4plus.pth",
        "repo": "ai-forever/Real-ESRGAN",
        "bytes": 67021162,
        "desc": "Real-ESRGAN (Limpeza de ruídos e nitidez geral suave)"
    }
}

print(f"📂 Diretório de Modelos no Google Drive: {UPSCALE_DIR}")
print("🔍 Verificando modelos...")

from huggingface_hub import hf_hub_download

for key, info in UPSCALE_MODELS.items():
    dest = UPSCALE_DIR / info["file"]
    if dest.is_file() and dest.stat().st_size == info["bytes"]:
        print(f"  [✅ JÁ EXISTE NO DRIVE] {info['desc']}")
    else:
        print(f"  [⬇️ BAIXANDO PARA O DRIVE] {info['desc']}...")
        downloaded = hf_hub_download(repo_id=info["repo"], filename=info["file"], local_dir=str(UPSCALE_DIR))
        p = Path(downloaded)
        if p != dest and p.is_file():
            shutil.move(str(p), str(dest))
        print(f"  [✅ CONCLUÍDO] {info['file']} salvo permanentemente no Drive.")

print("\n🎉 Todos os modelos de super-resolução estão prontos no Drive!")


## 🚀 Executar Upscaling de Vídeos (Individual ou Fila em Lote)

Configure os parâmetros abaixo no formulário:
* **`MODO_EXECUCAO`:**
  * `Arquivo_Unico`: Processa o vídeo indicado em `CAMINHO_VIDEO`.
  * `Fila_Em_Lote_Pasta`: Processa todos os arquivos `.mp4` dentro de `PASTA_DE_ENTRADA`.
* **`MODELO`:** Selecione `4x-UltraSharp` para máximo realismo.
* **`RESOLUCAO_FINAL`:** Escolha `1080p (Full HD)` (ideal para internet/redes), `2x` ou `4x`.
* **`PRESERVAR_AUDIO`:** Mantém a trilha de áudio original perfeitamente sincronizada.


In [ ]:
# =============================================================================
# 3. Processador de Vídeo com Super-Resolução Neural e Áudio
# =============================================================================
# @title 🎬 Configuração e Execução do Upscale { run: "auto" }
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path
import cv2
import torch
from tqdm.auto import tqdm
from spandrel import ModelLoader
from base64 import b64encode
from IPython.display import HTML, display

# --- Parâmetros Interativos do Formulário ---
MODO_EXECUCAO = "Arquivo_Unico"  # @param ["Arquivo_Unico", "Fila_Em_Lote_Pasta"]
CAMINHO_VIDEO = "/content/drive/MyDrive/H3/MiniMax-H3-Native-Colab/video"  # @param {type:"string"}
PASTA_DE_ENTRADA = "/content/drive/MyDrive/Videos_Para_Upscale"  # @param {type:"string"}
PASTA_DE_SAIDA = "/content/drive/MyDrive/Videos_Upscaled"  # @param {type:"string"}

MODELO_ESCOLHIDO = "4x-UltraSharp"  # @param ["4x-UltraSharp", "RealESRGAN_x4plus"]
RESOLUCAO_FINAL = "1080p (Full HD)"  # @param ["1080p (Full HD)", "2x", "4x"]
PRESERVAR_AUDIO = True  # @param {type:"boolean"}
QUALIDADE_CRF = 17  # @param {type:"slider", min:14, max:24, step:1}

# 1. Configurar pastas de saída
OUTPUT_DIR = Path(PASTA_DE_SAIDA)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Coletar lista de vídeos para processar
videos_to_process = []
if MODO_EXECUCAO == "Arquivo_Unico":
    target = Path(CAMINHO_VIDEO)
    if target.is_dir():
        # Pega o arquivo de vídeo mais recente dentro da pasta
        mp4s = list(target.glob("*.mp4"))
        if mp4s:
            latest = max(mp4s, key=lambda p: p.stat().st_mtime)
            videos_to_process.append(latest)
            print(f"ℹ️ Caminho aponta para diretório. Selecionado o vídeo mais recente: {latest.name}")
        else:
            raise RuntimeError(f"Nenhum arquivo .mp4 encontrado em {target}")
    elif target.is_file():
        videos_to_process.append(target)
    else:
        raise RuntimeError(f"Arquivo ou pasta não encontrada: {target}")
else:
    in_dir = Path(PASTA_DE_ENTRADA)
    if not in_dir.is_dir():
        in_dir.mkdir(parents=True, exist_ok=True)
        print(f"📁 Pasta de entrada criada em: {in_dir}")
        print("   Coloque seus vídeos de 10s a 30s dentro dessa pasta no seu Drive e execute a célula novamente.")
    else:
        mp4s = sorted(list(in_dir.glob("*.mp4")))
        if not mp4s:
            print(f"⚠️ Nenhum vídeo .mp4 encontrado em {in_dir}")
        videos_to_process = mp4s

if not videos_to_process:
    print("Nenhum vídeo para processar no momento.")
else:
    print(f"🎯 Total de vídeos na fila: {len(videos_to_process)}")
    for i, v in enumerate(videos_to_process, 1):
        print(f"   [{i}] {v.name} ({v.stat().st_size / (1024**2):.1f} MiB)")

    # 3. Carregar modelo de super-resolução na GPU Tesla T4
    model_file = UPSCALE_DIR / UPSCALE_MODELS[MODELO_ESCOLHIDO]["file"]
    print(f"\n🧠 Carregando modelo {model_file.name} na GPU...")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    loader = ModelLoader()
    upscale_model = loader.load_from_file(str(model_file)).eval().to(device)
    if torch.cuda.is_available():
        upscale_model = upscale_model.half()

    # 4. Processar cada vídeo da fila
    for v_idx, video_path in enumerate(videos_to_process, 1):
        print("\n" + "=" * 76)
        print(f"🎬 Processando [{v_idx}/{len(videos_to_process)}]: {video_path.name}")
        print("=" * 76)
        
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"❌ Erro ao abrir {video_path.name}, pulando...")
            continue
            
        in_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        in_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_s = total_frames / fps
        
        # Calcular dimensões de saída
        if RESOLUCAO_FINAL == "2x":
            out_w, out_h = in_w * 2, in_h * 2
        elif RESOLUCAO_FINAL == "4x":
            out_w, out_h = in_w * 4, in_h * 4
        else:  # 1080p
            out_h = 1080
            out_w = int(round((in_w * (1080 / in_h)) / 2) * 2)
            
        print(f"📊 Original: {in_w}x{in_h} @ {fps:.1f} fps ({duration_s:.1f}s | {total_frames} frames)")
        print(f"🚀 Saída   : {out_w}x{out_h} ({RESOLUCAO_FINAL})")
        
        # Extrair trilha de áudio original com precisão de timestamp
        temp_audio = Path(f"/content/temp_audio_{v_idx}.aac")
        has_audio = False
        if PRESERVAR_AUDIO:
            try:
                subprocess.run([
                    "ffmpeg", "-y", "-i", str(video_path), "-vn", "-c:a", "copy", str(temp_audio)
                ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
                if temp_audio.is_file() and temp_audio.stat().st_size > 1000:
                    has_audio = True
                    print(f"🎵 Trilha de áudio estéreo preservada!")
            except Exception as e:
                print(f"⚠️ Aviso na extração de áudio: {e}")
                
        # Configurar pipeline de codificação FFmpeg via pipe stdin (zero imagens soltas em disco)
        out_filename = f"{video_path.stem}_upscaled_{out_w}x{out_h}.mp4"
        out_path = OUTPUT_DIR / out_filename
        
        ffmpeg_cmd = [
            "ffmpeg", "-y",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-s", f"{out_w}x{out_h}",
            "-pix_fmt", "bgr24",
            "-r", str(fps),
            "-i", "-"
        ]
        
        if has_audio:
            ffmpeg_cmd.extend(["-i", str(temp_audio), "-c:a", "copy"])
            
        ffmpeg_cmd.extend([
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-crf", str(QUALIDADE_CRF),
            "-preset", "fast",
            "-movflags", "+faststart",
            str(out_path)
        ])
        
        ffmpeg_proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
        
        t0 = time.perf_counter()
        try:
            with tqdm(total=total_frames, desc=f"Upscaling ({video_path.stem[:18]})", unit="frame") as pbar:
                while True:
                    ret, frame = cap.read()
                    if not ret or frame is None:
                        break
                        
                    t_frame = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float() / 255.0
                    t_frame = t_frame.to(device=device)
                    if torch.cuda.is_available():
                        t_frame = t_frame.half()
                        
                    with torch.no_grad():
                        sr_tensor = upscale_model(t_frame)
                        
                    sr_frame = (sr_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1) * 255.0).byte().cpu().numpy()
                    
                    if (sr_frame.shape[1], sr_frame.shape[0]) != (out_w, out_h):
                        sr_frame = cv2.resize(sr_frame, (out_w, out_h), interpolation=cv2.INTER_AREA if sr_frame.shape[1] > out_w else cv2.INTER_LANCZOS4)
                        
                    ffmpeg_proc.stdin.write(sr_frame.tobytes())
                    pbar.update(1)
        finally:
            cap.release()
            if ffmpeg_proc.stdin:
                ffmpeg_proc.stdin.close()
            ffmpeg_proc.wait()
            if temp_audio.is_file():
                temp_audio.unlink(missing_ok=True)
                
        elapsed = time.perf_counter() - t0
        print(f"\n✅ Concluído em {elapsed:.1f}s ({total_frames / max(elapsed, 0.001):.1f} fps)")
        print(f"📁 Salvo em: {out_path} ({out_path.stat().st_size / (1024**2):.1f} MiB)")
        
        # Player inline para o último vídeo processado
        if v_idx == len(videos_to_process) and out_path.stat().st_size < 35 * 1024 * 1024:
            try:
                _data_url = "data:video/mp4;base64," + b64encode(out_path.read_bytes()).decode()
                display(HTML(f'''
                <div style="text-align: center; margin-top: 15px;">
                    <h4>🎬 Pré-visualização: {out_filename}</h4>
                    <video width="{min(out_w, 860)}" controls autoplay loop style="border-radius: 8px; max-width: 100%;">
                        <source src="{_data_url}" type="video/mp4">
                    </video>
                </div>
                '''))
            except Exception:
                pass

    print("\n🎉 Todos os vídeos da fila foram upscaled com sucesso!")
    print(f"📂 Verifique sua pasta no Drive: {OUTPUT_DIR}")
